In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.arima.model import ARIMA as ARIMA_Model
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

data = pd.read_csv('data/sp500_dataset.csv', index_col='Date', parse_dates=True)

data['Target'] = data['Close'].shift(-1)
data = data.dropna()

features = ['Open', 'High', 'Low', 'Close', 'Volume', 
            'SMA_50', 'SMA_200', 'MACD', 'Signal', 'RSI', 'Momentum']

X = data[features].values
y = data['Target'].values
dates = data.index.to_numpy()

LOOKBACK_WINDOW = 20

def create_sequences_with_dates(X, y, dates, window_size):
    X_seq, y_seq, y_dates = [], [], []
    for i in range(len(X) - window_size):
        X_seq.append(X[i:(i + window_size)])
        y_seq.append(y[i + window_size])
        y_dates.append(dates[i + window_size])
    return (np.array(X_seq, dtype=np.float32),
            np.array(y_seq, dtype=np.float32),
            np.array(y_dates, dtype='datetime64[ns]'))

X_seq, y_seq, dates_seq = create_sequences_with_dates(X, y, dates, LOOKBACK_WINDOW)

TEST_SIZE = 0.20
test_split_index = int(len(X_seq) * (1 - TEST_SIZE))
X_train_raw, X_test_raw = X_seq[:test_split_index], X_seq[test_split_index:]
y_train_raw, y_test_raw = y_seq[:test_split_index], y_seq[test_split_index:]
dates_test = dates_seq[test_split_index:]

print("Dataset shapes after preparation:")
print(f"  X_train: {X_train_raw.shape}, y_train: {y_train_raw.shape}")
print(f"  X_test:  {X_test_raw.shape}, y_test:  {y_test_raw.shape}")
print(f"  Test dates: {dates_test.shape} ({dates_test[0]} to {dates_test[-1]})")

close_idx = features.index('Close')
y_rw_pred = X_test_raw[:, -1, close_idx]

mse_rw = mean_squared_error(y_test_raw, y_rw_pred)
rmse_rw = np.sqrt(mse_rw)
mae_rw = mean_absolute_error(y_test_raw, y_rw_pred)
r2_rw = r2_score(y_test_raw, y_rw_pred)
mape_rw = np.mean(np.abs((y_test_raw - y_rw_pred) / y_test_raw)) * 100

print("\n" + "=" * 60)
print("RANDOM WALK BENCHMARK (Naive: Close_{t+1} = Close_t)")
print("=" * 60)
print(f"  MSE:  {mse_rw:.2f}")
print(f"  RMSE: {rmse_rw:.2f}")
print(f"  MAE:  {mae_rw:.2f}")
print(f"  MAPE: {mape_rw:.2f}%")
print(f"  R2:   {r2_rw:.4f}")

print("\n" + "=" * 60)
print("ARIMA BENCHMARK - REGRESSION")
print("=" * 60)

test_start_date = pd.Timestamp(dates_test[0])
train_close_arima = data.loc[data.index <= test_start_date, 'Close'].values

best_aic, best_order = np.inf, (5, 1, 0)
for p in [1, 3, 5]:
    for q in [0, 1]:
        try:
            tmp = ARIMA_Model(train_close_arima, order=(p, 1, q)).fit()
            if tmp.aic < best_aic:
                best_aic = tmp.aic
                best_order = (p, 1, q)
        except:
            pass

print(f"ARIMA Order: {best_order} (AIC={best_aic:.1f})")

fit_arima = ARIMA_Model(train_close_arima, order=best_order).fit()
arima_preds = []

for k in range(len(y_test_raw)):
    forecast = fit_arima.forecast(steps=1)[0]
    arima_preds.append(forecast)
    fit_arima = fit_arima.append([y_test_raw[k]], refit=False)
    if (k + 1) % 50 == 0 or (k + 1) == len(y_test_raw):
        print(f"  Progress: {k + 1}/{len(y_test_raw)}")

arima_preds = np.array(arima_preds)

mse_ar = mean_squared_error(y_test_raw, arima_preds)
rmse_ar = np.sqrt(mse_ar)
mae_ar = mean_absolute_error(y_test_raw, arima_preds)
r2_ar = r2_score(y_test_raw, arima_preds)
mape_ar = np.mean(np.abs((y_test_raw - arima_preds) / y_test_raw)) * 100

print("\nARIMA Results:")
print(f"  MSE:  {mse_ar:.2f}")
print(f"  RMSE: {rmse_ar:.2f}")
print(f"  MAE:  {mae_ar:.2f}")
print(f"  MAPE: {mape_ar:.2f}%")
print(f"  R2:   {r2_ar:.4f}")

print("\n" + "=" * 60)
print("STATISTICAL BENCHMARKS SUMMARY: Random Walk vs ARIMA")
print("=" * 60)
print(f"{'Metric':<10} {'Random Walk':>15} {'ARIMA':>15}")
print("-" * 42)
print(f"{'MSE':<10} {mse_rw:>15.2f} {mse_ar:>15.2f}")
print(f"{'RMSE':<10} {rmse_rw:>15.2f} {rmse_ar:>15.2f}")
print(f"{'MAE':<10} {mae_rw:>15.2f} {mae_ar:>15.2f}")
print(f"{'MAPE':<10} {mape_rw:>15.2f}% {mape_ar:>15.2f}%")
print(f"{'R2':<10} {r2_rw:>15.4f} {r2_ar:>15.4f}")
